# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html). Feel free to use other functions from that library.

In [2]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [3]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [4]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [5]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [6]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [7]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   null|     BE|   null|    null|      1|  null|   269|  6|    69| null|       1|    null|    0.0|    null|    null|    null|    null|    null|    null|    null|
|3070802| 1963| 1096|   null|     US|     TX|    null|      1|  null|     2|  6|    63| null|       0|    null|   null|    null|    null|    null|    null|    null|    null|    null|
|3070803| 1963| 1096|   null|     US|     IL|    null|      1|  null|     2|  6|    6

## Cleaning the data first

In [8]:
# Checking for citations table column names.

print(citations.columns)  

['CITING', 'CITED']


In [9]:
# Checking for patents table column names.

print(patents.columns) 

['PATENT', 'GYEAR', 'GDATE', 'APPYEAR', 'COUNTRY', 'POSTATE', 'ASSIGNEE', 'ASSCODE', 'CLAIMS', 'NCLASS', 'CAT', 'SUBCAT', 'CMADE', 'CRECEIVE', 'RATIOCIT', 'GENERAL', 'ORIGINAL', 'FWDAPLAG', 'BCKGTLAG', 'SELFCTUB', 'SELFCTLB', 'SECDUPBD', 'SECDLWBD']


In [10]:
# Here I'm cleaning the citations table. Replacing Nulls in 'CITING' and 'CITED' columns with 0.

cleaned_citations = citations.na.fill({
    "CITING": 0,  
    "CITED": 0     
})

In [11]:
# Here I'm displaying the first 5 rows of the cleaned citation table.

cleaned_citations.show(5) 

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [12]:
# Here I'm cleaning the patents table. Replacing Nulls in 'PATENT' with 0 and 'ASSIGNEE', 'COUNTRY' having Null values with unknown values..

cleaned_patents = patents.na.fill({
    "PATENT": 0,           
    "ASSIGNEE": "unknown", 
    "COUNTRY": "unknown",  
})

In [13]:
# Here I'm displaying the first 5 rows of the cleaned patent table.

cleaned_patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   null|     BE|   null|    null|      1|  null|   269|  6|    69| null|       1|    null|    0.0|    null|    null|    null|    null|    null|    null|    null|
|3070802| 1963| 1096|   null|     US|     TX|    null|      1|  null|     2|  6|    63| null|       0|    null|   null|    null|    null|    null|    null|    null|    null|    null|
|3070803| 1963| 1096|   null|     US|     IL|    null|      1|  null|     2|  6|    6

### A left join is being performed between two DataFrames, cleaned_citations and cleaned_patents. The join condition is based on the "CITING" column in cleaned_citations matching the "PATENT" column in cleaned_patents.

### cache(): This caches the result of the join in memory for more efficient retrieval in later operations.

In [14]:
Citing_State = cleaned_citations.join(cleaned_patents, cleaned_citations["CITING"] == cleaned_patents["PATENT"], how="left").cache()

### This selects three columns from the joined DataFrame: "CITED", "CITING", "POSTATE" as "CITING_STATE" resulting in the new DataFrame Citing_State which contains three columns: CITED, CITING, and CITING_STATE.

In [15]:
Citing_State = Citing_State.select("CITED", "CITING", col("POSTATE").alias("CITING_STATE"))

### Here this line of code displays the first 20 rows of the Citing_State DataFrame.

In [16]:
Citing_State.show()

+-------+-------+------------+
|  CITED| CITING|CITING_STATE|
+-------+-------+------------+
|1331793|3858258|          CA|
|1540798|3858258|          CA|
| 924225|3858527|        null|
|2444326|3858527|        null|
|2705120|3858527|        null|
|2967080|3858527|        null|
|3602157|3858527|        null|
|3638586|3858527|        null|
|3699902|3858527|        null|
| 957631|3858560|          IN|
|3675252|3858597|          MT|
|3815160|3858597|          MT|
|2290722|3858770|          CA|
|2777621|3858770|          CA|
|2782969|3858770|          CA|
|3040941|3858770|          CA|
| 982044|3859029|          NY|
|1020004|3859029|          NY|
|1830227|3859029|          NY|
|2752631|3859029|          NY|
+-------+-------+------------+
only showing top 20 rows



### This is a left join operation between two DataFrames: cleaned_citations and cleaned_patents.
### The join condition is that the column "CITING" in the cleaned_citations DataFrame must match the column "PATENT" in the cleaned_patents DataFrame.
### cache() - This will store the result of the join operation in memory (cache) to optimize performance for any future actions that reference Citing_State. Caching is useful when the same DataFrame is used multiple times, so it avoids recalculating it from scratch each time.

In [17]:
tempStep = Citing_State.join(cleaned_patents, Citing_State["CITED"] == cleaned_patents["PATENT"], how="left").cache()

### Here this statement selects specific columns from the Citing_State DataFrame. "CITED" and "CITING" are columns from the cleaned_citations DataFrame.

### col("POSTATE").alias("CITING_STATE"): This selects the POSTATE column (likely representing the state where the patent was filed or originated) from the cleaned_patents DataFrame, and renames it as "CITING_STATE" for clarity.

### The result we get is the new Citing_State DataFrame contains three columns: CITED, CITING, CITING_STATE

In [18]:
tempStep = tempStep.select("CITING", "CITING_STATE", "CITED", col("POSTATE").alias("CITED_STATE"))

### Here I display the first 20 rows of the Citing_State DataFrame to the console, allowing to inspect the results of the operations.

In [19]:
tempStep.show()

+-------+------------+-----+-----------+
| CITING|CITING_STATE|CITED|CITED_STATE|
+-------+------------+-----+-----------+
|4305315|          MN| 2366|       null|
|4192521|        null| 2366|       null|
|4253355|          MN| 2366|       null|
|5580635|          WI| 5156|       null|
|4976561|        null| 5518|       null|
|4480374|          MN| 5803|       null|
|5123817|        null| 6620|       null|
|4115020|        null| 7240|       null|
|4727698|          CA| 7253|       null|
|4360982|          IA| 7340|       null|
|4108250|          IL| 7340|       null|
|5692807|          PA|10817|       null|
|5581904|        null|11458|       null|
|4282613|          MI|12940|       null|
|4741426|          NY|13840|       null|
|4705153|          NY|13840|       null|
|4556218|          FL|14832|       null|
|4896714|        null|15447|       null|
|5065652|          OH|15790|       null|
|5058476|          OH|15790|       null|
+-------+------------+-----+-----------+
only showing top

### Here I'm applying a filter to the tempStep DataFrame. 

### col("CITING_STATE").isNotNull(): This checks that the value in the CITING_STATE column is not null.

### col("CITED_STATE").isNotNull(): This checks that the value in the CITED_STATE column is not null.

### And applying a logical "AND" operator. Both conditions (for CITING_STATE and CITED_STATE) need to be true for a row to be kept. Resulting in the DataFrame tempStep which only includes rows where neither the CITING_STATE nor the CITED_STATE is null. Any row where either of these columns had a null value is been removed.

In [20]:
tempStep = tempStep.filter(col("CITING_STATE").isNotNull() & col("CITED_STATE").isNotNull())

### Here I applied another filter to the DataFrame tempStep which keeps only rows where the value in the CITING_STATE column is the same as the value in the CITED_STATE column. 

### In other words, it filters out rows where the citing and cited patents are from different states resulting in he DataFrame which includes only rows where both CITING_STATE and CITED_STATE have the same value, i.e., where the patent citations are intra-state (from the same state).

In [21]:
tempStep = tempStep.filter(col("CITING_STATE") == col("CITED_STATE"))

### Here I display the first 20 rows of the Citing_State DataFrame to the console.

In [22]:
tempStep.show()

+-------+------------+-------+-----------+
| CITING|CITING_STATE|  CITED|CITED_STATE|
+-------+------------+-------+-----------+
|3861359|          IL|3072100|         IL|
|3917094|          WI|3072274|         WI|
|4051847|          CA|3077191|         CA|
|4041217|          MD|3079454|         MD|
|4385248|          NY|3079519|         NY|
|4945561|          NY|3081464|         NY|
|4884717|          MI|3086674|         MI|
|4053105|          MA|3087676|         MA|
|3893618|          MA|3087676|         MA|
|4249365|          IA|3088262|         IA|
|5247786|          IA|3088262|         IA|
|5035582|          PA|3089008|         PA|
|3884773|          NJ|3089888|         NJ|
|3935741|          TX|3090232|         TX|
|4828608|          NY|3093475|         NY|
|5173632|          NH|3094640|         NH|
|4650499|          OK|3097519|         OK|
|4553985|          OK|3097519|         OK|
|4684473|          NJ|3102098|         NJ|
|5956831|          CA|3102333|         CA|
+-------+--

### Here I grouped the data in the tempStep DataFrame by the CITING column. For each citing patent (i.e., each unique value in the CITING column), it counts how many times it appears in the DataFrame.

### Resulting in the count of occurrences is displayed for inspection. It displays the first 20 rows of the Citing_Count DataFrame in the console. It helps inspect how many times each citing patent appears in the dataset.

In [23]:
Citing_Count = tempStep.groupby("CITING").count()

Citing_Count.show()

+-------+-----+
| CITING|count|
+-------+-----+
|5300411|    7|
|3956677|    1|
|4171110|    4|
|4031936|    2|
|3894399|    3|
|4761449|    1|
|4369612|    3|
|4052003|    4|
|4151539|    2|
|4414386|    1|
|4868906|    2|
|4339036|    2|
|4668726|   13|
|5279306|    7|
|4608206|    5|
|4673168|    2|
|4923281|    8|
|5912450|   15|
|4559622|    7|
|4841690|    4|
+-------+-----+
only showing top 20 rows



### Here this below line code imports the col function from PySpark's SQL functions. 

### col is used to refer to DataFrame columns by name in various operations like filtering, selecting, and ordering.

In [24]:
from pyspark.sql.functions import col

### Here this line of code performs a left join between the cleaned_patents DataFrame and the Citing_Count DataFrame. This join is based on the condition where the "PATENT" column in the cleaned_patents DataFrame matches the "CITING" column in the Citing_Count DataFrame.

### Here the left join keeps all rows from the cleaned_patents DataFrame, and only the matching rows from Citing_Count. If there is no match, null values will be returned for columns from Citing_Count.

### cache(): This caches the result of the join in memory for faster access in subsequent operations.

In [25]:
finalStep = cleaned_patents.join(Citing_Count, cleaned_patents["PATENT"] == Citing_Count["CITING"], how="left").cache()

### Here it sorts the rows of the finalStep DataFrame by the count column in descending order. The count column represents how many times a citing patent appeared in the same state as the cited patent (from the previous steps).

### Resulting in the DataFrame which is ordered so that patents with the highest countsare at the top.

In [26]:
finalStep = finalStep.orderBy(col("count"), ascending=False)

### Here it selects a subset of columns from the finalStep DataFrame for the final output. Columns such as "PATENT", "GYEAR", "POSTATE", "ASSIGNEE", etc., are selected for the output.

### The resulting DataFrame has all the necessary columns, with count renamed to "SAME_STATE" to indicate how many times a patent was cited by another patent in the same state.


In [27]:
finalStep = finalStep.select(
    "PATENT", "GYEAR", "GDATE", "APPYEAR", "COUNTRY", "POSTATE", "ASSIGNEE", 
    "ASSCODE", "CLAIMS", "NCLASS", "CAT", "SUBCAT", "CMADE", "CRECEIVE", 
    "RATIOCIT", "GENERAL", "ORIGINAL", "FWDAPLAG", "BCKGTLAG", "SELFCTUB", 
    "SELFCTLB", "SECDUPBD", "SECDLWBD", col("count").alias("SAME_STATE")  # Rename 'count' to 'SAME_STATE'
)

### This line of code displays the first 10 rows of the finalStep DataFrame.

### 'truncate=False' ensures that none of the columns are truncated in the output.

In [28]:
finalStep.show(10, truncate=False)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|PATENT |GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|SAME_STATE|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|5959466|1999 |14515|1997   |US     |CA     |5310    |2      |null  |326   |4  |46    |159  |0       |1.0     |null   |0.6186  |null    |4.8868  |0.0455  |0.044   |null    |null    |125       |
|5983822|1999 |14564|1998   |US     |TX     |569900  |2      |null  |114   |5  |55    |200  |0       |0.995   |null   |0.7201  |null    |12.45   |0.0     |0.0     |null    |null    |103       |
|6008204|1999 |14606|1998   |U